In [73]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d

ROOT_DIR = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "src").is_dir() and (directory / "data").is_dir()
)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

PROCESSED_DIR = ROOT_DIR / "data/processed"
REPORT_IMAGES_DIR = ROOT_DIR / "report/report_images"

from src.utils.plotting import show_geometries, save_open3d_screenshot

## Paso 0 - cargar nube global

In [74]:
ply_path = PROCESSED_DIR / "palacio" / "nube_global_palacio.ply"
pcd = o3d.io.read_point_cloud(str(ply_path))

n_inicial = len(pcd.points)
print(f"Puntos iniciales: {n_inicial:,}")

# registro de puntos sobrevivientes por etapa (para el informe)
conteos = {"inicial": n_inicial}

show_geometries([pcd])

Puntos iniciales: 87,150,016


## Paso 1 - Filtrado espacial (recorte de caja)

In [75]:
puntos = np.asarray(pcd.points)
print("min (x,y,z):", puntos.min(axis=0))
print("max (x,y,z):", puntos.max(axis=0))

min (x,y,z): [-3417.86434274 -3420.34731459 -1004.69847093]
max (x,y,z): [3583.41896217 3557.01828882 2608.55673503]


In [76]:
min_bound = np.array([130.0, 90.0, -140.0])  
max_bound = np.array([ 220.0,  470.0,  0.0])    
bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
bbox.color = (1, 0, 0)

pcd_bbox = pcd.crop(bbox)

conteos["bbox"] = len(pcd_bbox.points)
print(f"Antes:   {len(pcd.points):,}")
print(f"Después: {conteos['bbox']:,}")

show_geometries([pcd_bbox])

Antes:   87,150,016
Después: 8,901,754


## Paso 2 - Submuestreo (voxel downsample)

In [77]:
voxel_size = 1.5   # mm — más chico conserva más detalle y sube el peso del .ply
pcd_voxel = pcd_bbox.voxel_down_sample(voxel_size=voxel_size)

conteos["voxel"] = len(pcd_voxel.points)
print(f"Tras voxel ({voxel_size} mm): {conteos['voxel']:,} puntos")
show_geometries([pcd_voxel])

Tras voxel (1.5 mm): 420,612 puntos


## Paso 3 - Eliminación de outliers

In [78]:
pcd_limpio, idx = pcd_voxel.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=1.5,
)

conteos["outliers"] = len(pcd_limpio.points)
print(f"Tras outliers: {conteos['outliers']:,} puntos")
show_geometries([pcd_limpio])

Tras outliers: 395,324 puntos


In [79]:
# version para informe
outliers = pcd_voxel.select_by_index(idx, invert=True)   
outliers.paint_uniform_color([1, 0, 0])
show_geometries([pcd_limpio, outliers])

### Puntos sobrevivientes por etapa

In [80]:
# Tabla de puntos por etapa (para el informe)
print(f"{'etapa':>10} | {'puntos':>12} | {'% del inicial':>13}")
print("-" * 42)
for etapa, n in conteos.items():
    print(f"{etapa:>10} | {n:>12,} | {100 * n / conteos['inicial']:>12.3f}%")

     etapa |       puntos | % del inicial
------------------------------------------
   inicial |   87,150,016 |      100.000%
      bbox |    8,901,754 |       10.214%
     voxel |      420,612 |        0.483%
  outliers |      395,324 |        0.454%


In [81]:
# Capturas de cada etapa para el informe
REPORT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

for nube, nombre in [
    (pcd_bbox, "palacio_bbox"),
    (pcd_voxel, "palacio_voxel"),
    (pcd_limpio, "palacio_optimizada"),
]:
    save_open3d_screenshot([nube], str(REPORT_IMAGES_DIR / f"{nombre}.png"))
    print("Guardada:", REPORT_IMAGES_DIR / f"{nombre}.png")

Guardada: c:\Users\Usuario\OneDrive\Documentos\UdeSA cursos de ingreso 2023\udesa ingenieria en IA\carrera\3°er año ingenieria en IA\2°do semestre\vision artificial\TPs\Vision-Artificial-TP2\report\report_images\palacio_bbox.png
Guardada: c:\Users\Usuario\OneDrive\Documentos\UdeSA cursos de ingreso 2023\udesa ingenieria en IA\carrera\3°er año ingenieria en IA\2°do semestre\vision artificial\TPs\Vision-Artificial-TP2\report\report_images\palacio_voxel.png
Guardada: c:\Users\Usuario\OneDrive\Documentos\UdeSA cursos de ingreso 2023\udesa ingenieria en IA\carrera\3°er año ingenieria en IA\2°do semestre\vision artificial\TPs\Vision-Artificial-TP2\report\report_images\palacio_optimizada.png


## Paso 4 - Exportar el .ply

In [82]:
salida = PROCESSED_DIR / "palacio" / "nube_palacio_optimizada.ply"
o3d.io.write_point_cloud(str(salida), pcd_limpio, write_ascii=False)   # binario
print("Guardado en:", salida)
print("Tamaño:", round(salida.stat().st_size / 1e6, 2), "MB")

Guardado en: c:\Users\Usuario\OneDrive\Documentos\UdeSA cursos de ingreso 2023\udesa ingenieria en IA\carrera\3°er año ingenieria en IA\2°do semestre\vision artificial\TPs\Vision-Artificial-TP2\data\processed\palacio\nube_palacio_optimizada.ply
Tamaño: 10.67 MB


## Paso 5 - Medición métrica

In [ ]:
aabb = pcd_limpio.get_axis_aligned_bounding_box()
dx, dy, dz = np.round(aabb.get_extent(), 1)
print("== AABB (ejes del mundo) ==")
print(f"  ancho  (x): {dx:7.1f} mm")
print(f"  largo  (y): {dy:7.1f} mm")
print(f"  alto   (z): {dz:7.1f} mm")

obb = pcd_limpio.get_minimal_oriented_bounding_box()
ex, ey, ez = np.round(np.sort(obb.extent)[::-1], 1)  # ordenadas de mayor a menor
print("\n== OBB (ajustada al objeto) ==")
print(f"  dimensiones (mayor->menor): {ex:.1f} x {ey:.1f} x {ez:.1f} mm")

alto_real_mm = None   # p. ej. 135.0
if alto_real_mm is not None:
    error_rel = 100 * abs(dz - alto_real_mm) / alto_real_mm
    print(f"\nAlto estimado: {dz:.1f} mm | real: {alto_real_mm:.1f} mm "
          f"| error relativo: {error_rel:.1f}%")

# Visualización de la medición
obb.color = (1, 0, 0)
show_geometries([pcd_limpio, obb])

== AABB (ejes del mundo) ==
  ancho  (x):    90.0 mm
  largo  (y):   379.8 mm
  alto   (z):   138.1 mm

== OBB (ajustada al objeto) ==
  dimensiones (mayor->menor): 379.8 x 138.1 x 90.0 mm
